In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# import os

# # Set project path
# project_path = (
#     '/content/drive/MyDrive/My_Remote_PC/Colab/Data_engineer'
# )
# external_source_path ="/external_source"

# # Change working directory (optional)
# os.chdir(project_path)

# # Verify contents
# print(os.listdir('.'))

['NorthStar.ipynb', 'data_engineering_capstone_project.pdf', 'NorthStar.drawio', 'olist_customers_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv', 'external_source']


In [ ]:
# !pip install kagglehub

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)

   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   -------- ------------------------------- 1/5 [requests]
   -------- ------------------------------- 1/5 [requests]
   -------- ------------------------------- 1/5 [requests]
   -------- ------------------------------

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

c:\Users\USER\Desktop\Personal\northstar\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 42.6M/42.6M [00:10<00:00, 4.15MB/s]

Extracting files...


Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2


In [3]:
external_source_path ="data/source"

In [4]:
import os
import shutil

# Ensure your target drive directory exists
os.makedirs(external_source_path, exist_ok=True)

# Copy downloaded files from cache into your Google Drive path
shutil.copytree(path, external_source_path, dirs_exist_ok=True)

print("Dataset copied to Google Drive:", external_source_path)
print("Files:", os.listdir(external_source_path))

Dataset copied to Google Drive: data/source
Files: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [ ]:
import numpy as np
import pandas as pd

In [1]:
a=5

In [2]:
import duckdb

con = duckdb.connect("data/northstar.duckdb")

tables = {
    "silver_customers": "data/silver/customers/data.parquet",
    "silver_sellers": "data/silver/sellers/data.parquet",
    "silver_products": "data/silver/products/data.parquet",
    "silver_orders": "data/silver/orders/data.parquet",
    "silver_order_items": "data/silver/order_items/data.parquet",
    "silver_order_payments": "data/silver/order_payments/data.parquet",
    "silver_order_reviews": "data/silver/order_reviews/data.parquet",
    "silver_geolocation": "data/silver/geolocation/data.parquet",
    "silver_order_events": "data/silver/order_events/data.parquet",
}

for table_name, parquet_path in tables.items():
    con.execute(f"""
        CREATE OR REPLACE VIEW {table_name} AS
        SELECT *
        FROM read_parquet('{parquet_path}')
    """)

print("Silver views created:")
for table_name in tables:
    print(f"  ✓ {table_name}")

print("\nView verification:")
print(con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
"""))

con.close()

Silver views created:
  ✓ silver_customers
  ✓ silver_sellers
  ✓ silver_products
  ✓ silver_orders
  ✓ silver_order_items
  ✓ silver_order_payments
  ✓ silver_order_reviews
  ✓ silver_geolocation
  ✓ silver_order_events

View verification:
┌───────────────────────┐
│      table_name       │
│        varchar        │
├───────────────────────┤
│ silver_customers      │
│ silver_geolocation    │
│ silver_order_events   │
│ silver_order_items    │
│ silver_order_payments │
│ silver_order_reviews  │
│ silver_orders         │
│ silver_products       │
│ silver_sellers        │
└───────────────────────┘



In [3]:
import duckdb

con = duckdb.connect("data/northstar.duckdb")

tables = [
    "silver_customers",
    "silver_sellers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "silver_order_payments",
    "silver_order_reviews",
    "silver_geolocation",
    "silver_order_events",
]

print("DuckDB Silver View Row Counts")
print("=" * 40)

for table in tables:
    count = con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<25} {count:>10,}")

con.close()

DuckDB Silver View Row Counts
silver_customers              99,441
silver_sellers                 3,095
silver_products               32,951
silver_orders                 99,441
silver_order_items           112,650
silver_order_payments        103,886
silver_order_reviews          99,224
silver_geolocation           738,332
silver_order_events               54


In [4]:

con = duckdb.connect("data/northstar.duckdb")

tables = [
    "silver_customers",
    "silver_sellers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "silver_order_payments",
    "silver_order_reviews",
    "silver_geolocation",
]

for table in tables:
    print("\n" + "=" * 70)
    print(table)
    print("=" * 70)

    result = con.sql(f"""
        DESCRIBE {table}
    """)

    print(result)

con.close()


silver_customers
┌──────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │       column_type        │  null   │   key   │ default │  extra  │
│         varchar          │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id              │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_unique_id       │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_zip_code_prefix │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_city            │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_state           │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ _ingestion_timestamp     │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ _source_file        

In [5]:

con = duckdb.connect("data/northstar.duckdb")

checks = {
    "fact_orders rows":
        "SELECT COUNT(*) FROM fact_orders",

    "fact_orders duplicate order_id":
        """
        SELECT COUNT(*) FROM (
            SELECT order_id
            FROM fact_orders
            GROUP BY order_id
            HAVING COUNT(*) > 1
        )
        """,

    "fact_order_items duplicate grain":
        """
        SELECT COUNT(*) FROM (
            SELECT order_id, order_item_id
            FROM fact_order_items
            GROUP BY order_id, order_item_id
            HAVING COUNT(*) > 1
        )
        """,

    "fact_payments duplicate grain":
        """
        SELECT COUNT(*) FROM (
            SELECT order_id, payment_sequential
            FROM fact_payments
            GROUP BY order_id, payment_sequential
            HAVING COUNT(*) > 1
        )
        """,

    "orders missing customer key":
        """
        SELECT COUNT(*)
        FROM fact_orders
        WHERE customer_key IS NULL
        """,

    "items missing product key":
        """
        SELECT COUNT(*)
        FROM fact_order_items
        WHERE product_key IS NULL
        """,

    "items missing seller key":
        """
        SELECT COUNT(*)
        FROM fact_order_items
        WHERE seller_key IS NULL
        """,

    "orders missing order date":
        """
        SELECT COUNT(*)
        FROM fact_orders
        WHERE order_date_key IS NULL
        """
}

print("Gold Model Validation")
print("=" * 60)

for name, query in checks.items():
    result = con.execute(query).fetchone()[0]

    if "rows" in name:
        status = "PASS" if result == 99441 else "CHECK"
    else:
        status = "PASS" if result == 0 else "FAIL"

    print(f"{status:<6} {name:<40} {result:,}")

con.close()

Gold Model Validation
PASS   fact_orders rows                         99,441
PASS   fact_orders duplicate order_id           0
PASS   fact_order_items duplicate grain         0
PASS   fact_payments duplicate grain            0
PASS   orders missing customer key              0
PASS   items missing product key                0
PASS   items missing seller key                 0
PASS   orders missing order date                0


In [11]:
import duckdb; con=duckdb.connect("data/northstar.duckdb"); 
print(con.execute("SELECT ''dim_customer'' AS table_name, COUNT(*) AS rows) FROM dim_customer UNION ALL SELECT ''dim_product'', COUNT(*) FROM dim_product UNION ALL SELECT ''dim_seller'', COUNT(*) FROM dim_seller UNION ALL SELECT ''dim_geography'', COUNT(*) FROM dim_geography UNION ALL SELECT ''dim_date'', COUNT(*) FROM dim_date").fetchdf().to_string(index=False)); 
con.close()

ParserException: Parser Error: syntax error at or near "''"

LINE 1: SELECT ''dim_customer'' AS table_name, COUNT(*) AS rows) FROM dim_customer UNION...
                             ^

In [12]:
import duckdb

con = duckdb.connect("data/northstar.duckdb")

for table in [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_geography",
    "dim_date"
]:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count:,}")

con.close()

dim_customer: 96,096
dim_product: 32,951
dim_seller: 3,095
dim_geography: 19,015
dim_date: 821


In [1]:
import duckdb

con = duckdb.connect("data/northstar.duckdb")

print("ROW COUNT")
print("-" * 40)

count = con.execute(
    "SELECT COUNT(*) FROM mart_sales_daily"
).fetchone()[0]

print(f"mart_sales_daily: {count:,}")


print("\nSAMPLE ROLLING METRICS")
print("-" * 40)

sample = con.execute("""
    SELECT
        full_date,
        daily_revenue,
        daily_orders,
        rolling_7d_revenue,
        rolling_7d_orders
    FROM mart_sales_daily
    WHERE daily_revenue > 0
    ORDER BY full_date
    LIMIT 10
""").fetchdf()

print(sample.to_string(index=False))


print("\nWINDOW VALIDATION")
print("-" * 40)

result = con.execute("""
    SELECT
        full_date,
        daily_revenue,
        rolling_7d_revenue,
        rolling_7d_orders
    FROM mart_sales_daily
    ORDER BY full_date
    LIMIT 10
""").fetchdf()

print(result.to_string(index=False))

con.close()

ROW COUNT
----------------------------------------
mart_sales_daily: 821

SAMPLE ROLLING METRICS
----------------------------------------
 full_date  daily_revenue  daily_orders  rolling_7d_revenue  rolling_7d_orders
2016-09-04         136.23             1              136.23                1.0
2016-09-05          75.06             1              211.29                2.0
2016-09-15         143.46             1              143.46                1.0
2016-10-02         109.34             1              109.34                1.0
2016-10-03         595.14             8              704.48                9.0
2016-10-04       11295.48            60            11999.96               69.0
2016-10-05        9645.94            42            21645.90              111.0
2016-10-06        9131.23            49            30777.13              160.0
2016-10-07        8135.97            45            38913.10              205.0
2016-10-08        9383.37            40            48296.47             